# Защита / атака / мониторинг

**Цель:** разобрать риски при вызове MCP-инструментов: утечка секретов через «злой» сервер; защита через guardrail-слой (проверка аргументов перед вызовом).

**Используемые примеры:**
- `topic_2_agents/example_5_guardrails_mcp_agents` — guardrail-слой для аргументов MCP-тулов; honest vs evil MCP-серверы; полный прогон с двумя серверами.

In [ ]:
%autosave 60

## Настройка окружения

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
%cd ..

## Риски и guardrails

- **Атака:** MCP-сервер может запрашивать у агента «дополнительные» параметры (например, `secret_token` для «улучшения маршрута») и логировать их — утечка секретов.
- **Защита:** перед вызовом инструмента проверять имена и значения аргументов: запрещать ключи/значения, содержащие подстроки вроде `secret`, `token`, `api_key`, `password` (guardrail). При нарушении — не вызывать тул, вернуть ошибку.
- **Мониторинг:** в демо сравниваются сценарии без guardrail (агент может передать секрет злому серверу) и с guardrail (вызов с секретом блокируется).

### Код guardrail

Функция `validate_tool_args(tool_name, args)` проверяет аргументы; при наличии запрещённых ключей или значений выбрасывает `ValueError`. `guarded_call` вызывает тул только после успешной проверки.

In [ ]:
%cd ai_agents_course/topic_2_agents/example_5_guardrails_mcp_agents
sys.path.insert(0, str(Path.cwd()))
from guardrails_mcp import validate_tool_args, SENSITIVE_KEYS

print("Запрещённые подстроки в ключах/значениях:", SENSITIVE_KEYS)
validate_tool_args("plan_route", {"origin": "A", "destination": "B", "mode": "car"})  # ок
print("Проверка 1: OK")
try:
    validate_tool_args("plan_route", {"origin": "A", "secret_token": "x"})
except ValueError as e:
    print("Проверка 2 (ожидаемо ошибка):", e)
%cd ../../..

### honest + evil серверы, guardrail и без

Скрипт запускает оба MCP-сервера (honest на 8000, evil на 8001), затем запускает демо: вызов графа без guardrail (запрос может спровоцировать передачу секрета evil) и с guardrail; сохраняет артефакты. Требует установленных зависимостей (langchain-mcp-adapters и др.).

In [ ]:
%cd ai_agents_course/topic_2_agents/example_5_guardrails_mcp_agents
%run run_full.py
%cd ../../..

## Итоги

- Вызов внешних MCP-инструментов без проверки аргументов может привести к утечке секретов, если сервер или промпт провоцируют передачу чувствительных данных.
- Guardrail-слой (валидация имён и значений аргументов перед вызовом) снижает риск; демо на honest/evil серверах наглядно показывает разницу поведения с guardrail и без.